<a href="https://colab.research.google.com/github/rakelcey/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Money Agent Network Clean-Up (Seed 44)

Generate Raw Data & Define clean_agent_data()

In [1]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)


def generate_agent_data() -> tuple[pd.DataFrame, pd.DataFrame]:
  """Generate mobile money agents and raw transactions (seed 44)."""
  rng = np.random.default_rng(44)
  n = 600
  agents = pd.DataFrame({
      "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
      "county": rng.choice(
          ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
      ),
      "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime(
          "%Y-%m-%d"
      ),
  })
  txns = pd.DataFrame({
      "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
      "agent_id": rng.choice(agents["agent_id"], n),
      "txn_type": rng.choice(
          ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
      ),
      "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
      "txn_date": (
          pd.to_datetime("2026-03-01")
          + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
      ).strftime("%Y-%m-%d"),
  })
  idx = rng.choice(n, 30, replace=False)
  txns.loc[idx[:15], "amount"] = np.nan
  txns.loc[idx[15:20], "amount"] = "N/A"
  txns.loc[idx[20:25], "amount"] = "250000"
  txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace(
      "-", "/"
  )
  txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
  txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

  agents.to_csv("agents.csv", index=False)
  txns.to_csv("agent_transactions.csv", index=False)
  return agents, txns


def clean_agent_data(
    txns: pd.DataFrame, agents: pd.DataFrame
) -> tuple[pd.DataFrame, dict]:
  """Clean raw agent transactions and enrich with agent county details."""
  df = txns.copy()
  before_rows = len(df)

  # 1. Remove duplicate txn_id rows (keep first)
  dup_mask = df.duplicated(subset=["txn_id"], keep="first")
  duplicates_removed = int(dup_mask.sum())
  df = df[~dup_mask].copy()

  # 2. Standardise txn_type to 'Deposit' or 'Withdrawal'
  df["txn_type"] = df["txn_type"].astype(str).str.strip().str.title()

  # 3. Convert amount to numeric, flag imputed values, and impute by txn_type median
  raw_numeric = pd.to_numeric(df["amount"], errors="coerce")
  df["amount_imputed"] = raw_numeric.isna()
  amount_imputed_count = int(df["amount_imputed"].sum())
  df["amount"] = raw_numeric

  type_medians = df.groupby("txn_type")["amount"].median()
  overall_median = float(df["amount"].median())
  df["amount"] = (
      df["amount"]
      .fillna(df["txn_type"].map(type_medians))
      .fillna(overall_median)
  )

  # 4. Parse txn_date into datetime64[ns]
  df["txn_date"] = pd.to_datetime(
      df["txn_date"], format="mixed", errors="coerce"
  )

  # 5. Flag outliers using 1.5 * IQR
  q1 = float(df["amount"].quantile(0.25))
  q3 = float(df["amount"].quantile(0.75))
  iqr = q3 - q1
  lower_fence = q1 - 1.5 * iqr
  upper_fence = q3 + 1.5 * iqr
  df["is_outlier"] = (df["amount"] < lower_fence) | (df["amount"] > upper_fence)

  # 6. Merge with agents on agent_id
  unknown_agent_txns = int((~df["agent_id"].isin(agents["agent_id"])).sum())
  clean_df = df.merge(
      agents[["agent_id", "county"]].drop_duplicates(subset=["agent_id"]),
      on="agent_id",
      how="left",
  )
  clean_df["county"] = clean_df["county"].fillna("Unknown")

  log = {
      "rows_before": before_rows,
      "duplicates_removed": duplicates_removed,
      "rows_after": len(clean_df),
      "amount_imputed_count": amount_imputed_count,
      "iqr_q1": round(q1, 2),
      "iqr_q3": round(q3, 2),
      "iqr_upper_fence": round(upper_fence, 2),
      "outliers_flagged": int(df["is_outlier"].sum()),
      "rows_at_250000": int((df["amount"] == 250000).sum()),
      "unknown_agent_txns": unknown_agent_txns,
  }
  return clean_df, log


agents, txns = generate_agent_data()
print("Raw agents:", agents.shape, "| Raw txns:", txns.shape)

Raw agents: (20, 3) | Raw txns: (612, 5)


Data-Quality Audit Summary

In [2]:
audit_summary = pd.DataFrame([
    {
        "issue": "Duplicate txn_id rows",
        "column": "txn_id",
        "count": int(txns.duplicated(subset=["txn_id"], keep="first").sum()),
    },
    {
        "issue": "Inconsistent casing/whitespace in txn_type",
        "column": "txn_type",
        "count": int((~txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    },
    {
        "issue": "Missing amount (NaN)",
        "column": "amount",
        "count": int(txns["amount"].isna().sum()),
    },
    {
        "issue": "Non-numeric 'N/A' in amount",
        "column": "amount",
        "count": int(
            (
                txns["amount"].notna()
                & pd.to_numeric(txns["amount"], errors="coerce").isna()
            ).sum()
        ),
    },
    {
        "issue": "Slash-formatted dates (YYYY/MM/DD)",
        "column": "txn_date",
        "count": int(
            txns["txn_date"].astype(str).str.contains("/", na=False).sum()
        ),
    },
    {
        "issue": "Unknown agent_id ('AG099')",
        "column": "agent_id",
        "count": int((~txns["agent_id"].isin(agents["agent_id"])).sum()),
    },
])
audit_summary

,issue,column,count
0,Duplicate txn_id rows,txn_id,12
1,Inconsistent casing/whitespace in txn_type,txn_type,384
2,Missing amount (NaN),amount,15
3,Non-numeric 'N/A' in amount,amount,7
4,Slash-formatted dates (YYYY/MM/DD),txn_date,5
5,Unknown agent_id ('AG099'),agent_id,8


Clean Data, Export CSV & Print Summaries

In [3]:
clean_df, log = clean_agent_data(txns, agents)

print("Cleaning Log:", log)
print(
    f"\nBefore cleaning: {log['rows_before']} rows | After merge:"
    f" {len(clean_df)} rows"
)

out_cols = [
    "txn_id",
    "agent_id",
    "county",
    "txn_type",
    "amount",
    "txn_date",
    "amount_imputed",
    "is_outlier",
]
clean_df[out_cols].to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved outputs/agent_transactions_clean.csv")

county_summary = (
    clean_df.groupby(["county", "txn_type"], as_index=False)
    .agg(
        total_value=("amount", "sum"),
        txn_count=("txn_id", "count"),
        avg_value=("amount", "mean"),
    )
    .sort_values(["county", "txn_type"])
)
county_summary["avg_value"] = county_summary["avg_value"].round(2)
print("\nCounty Summary by Transaction Type:")
print(county_summary.to_string(index=False))

clean_df["week"] = clean_df["txn_date"].dt.to_period("W-SUN").astype(str)
weekly_pivot = clean_df.pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
weekly_pivot["Total"] = weekly_pivot.sum(axis=1)
print("\nWeekly Pivot by County:")
weekly_pivot

Cleaning Log: {'rows_before': 612, 'duplicates_removed': 12, 'rows_after': 600, 'amount_imputed_count': 20, 'iqr_q1': 1105.0, 'iqr_q3': 2868.5, 'iqr_upper_fence': 5513.75, 'outliers_flagged': 53, 'rows_at_250000': 5, 'unknown_agent_txns': 7}

Before cleaning: 612 rows | After merge: 600 rows
Saved outputs/agent_transactions_clean.csv

County Summary by Transaction Type:
 county   txn_type  total_value  txn_count  avg_value
 Kiambu    Deposit     334737.0         43    7784.58
 Kiambu Withdrawal      77024.5         27    2852.76
 Kisumu    Deposit     436501.5         74    5898.67
 Kisumu Withdrawal      76348.5         37    2063.47
Mombasa    Deposit     149094.0         72    2070.75
Mombasa Withdrawal     368592.0         61    6042.49
Nairobi    Deposit     543698.0        118    4607.61
Nairobi Withdrawal     161932.0         62    2611.81
 Nakuru    Deposit     184944.0         60    3082.40
 Nakuru Withdrawal     328568.5         39    8424.83
Unknown    Deposit      25279.0  

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03,Total
county,,,,,,,,,,,
Kiambu,0.0,18317.0,267745.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0,411761.5
Kisumu,1893.0,287933.0,23186.0,28126.0,27077.0,41077.0,15341.0,29470.0,37439.0,21308.0,512850.0
Mombasa,6759.0,285696.5,30504.5,25171.5,17050.0,38344.0,47119.5,26730.0,16687.0,23624.0,517686.0
Nairobi,7046.0,68967.0,52887.5,43134.5,41400.5,282931.0,56813.5,56406.5,65135.0,30908.5,705630.0
Nakuru,2770.5,56574.0,28730.0,39795.0,18972.5,18145.0,276609.0,26179.5,30862.0,14875.0,513512.5
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0,27156.0


## Data Quality Report

* **Completeness:** Missing (`NaN`) and `"N/A"` transaction amounts were flagged in `amount_imputed` and imputed using the median amount per `txn_type` (`Deposit` or `Withdrawal`) so records were preserved without distorting skewed totals.
* **Uniqueness:** Duplicate `txn_id` rows (`612` raw rows reduced to `600` unique rows) were removed keeping the first record so transactions are not double-counted.
* **Timeliness:** Transaction dates span `2026-03-01` to `2026-04-29` across a 60-day window and group cleanly into weekly reporting buckets (`W-SUN`).
* **Validity:** Non-numeric `"N/A"` strings and slash-delimited dates (`YYYY/MM/DD`) were safely coerced/parsed into numeric and `datetime64[ns]` types.
* **Accuracy:** High-value transactions (including `250000` entries above the `1.5 * IQR` upper fence) are flagged in `is_outlier = True` for operational review rather than silently deleted.
* **Consistency & Privacy:** `txn_type` values were standardized to `"Deposit"` and `"Withdrawal"`, and unknown agent `"AG099"` rows were retained via a left join with `county = "Unknown"`. Under the **Kenya Data Protection Act (2019)**, if customer phone numbers were included, they would be treated as personal data requiring data minimization, pseudonymisation/hashing, access controls, and purpose limitation.